### Read data set

In [4]:
import pandas as pd

In [5]:
df=pd.read_csv(r'../resource/customer_support_tickets_200k.csv')

In [6]:
df.head()

,ticket_id,customer_name,customer_email,product,category,issue_description,resolution_notes,priority,status,channel,...,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,issue_complexity_score,customer_segment
0,1,Patricia Smith,patricia.smith760@outlook.com,Web Portal,Account Suspension,The payment was deducted from my bank account ...,Data synchronization restored after backend se...,Urgent,Open,Email,...,2023-05-20,No,Yes,MacOS,Edge,PayPal,French,Afternoon,4,Small Business
1,2,Patricia Williams,patricia.williams390@gmail.com,Mobile App,Performance Issue,I found a bug in the latest update affecting r...,Provided step-by-step troubleshooting instruct...,Urgent,Closed,Email,...,2024-01-19,Yes,Yes,Windows,Firefox,PayPal,English,Afternoon,2,Small Business
2,3,William Anderson,william.anderson651@outlook.com,Web Portal,Performance Issue,The application crashes whenever I try to uplo...,Provided step-by-step troubleshooting instruct...,Medium,Closed,Chat,...,2022-12-05,Yes,Yes,Windows,Safari,Bank Transfer,French,Morning,4,Corporate
3,4,David Miller,david.miller672@icloud.com,Payment Gateway,Subscription Cancellation,My subscription was cancelled without my reque...,Provided step-by-step troubleshooting instruct...,Medium,Closed,Social Media,...,2024-04-04,Yes,No,Windows,Chrome,Credit Card,Spanish,Afternoon,7,Corporate
4,5,Robert Gonzalez,robert.gonzalez391@hotmail.com,Web Portal,Feature Request,The system is not syncing data across devices ...,We have reset the account credentials and advi...,High,Pending Customer,Email,...,2024-08-24,Yes,No,Linux,NaN,Debit Card,Spanish,Evening,3,Corporate


### Feature Selection

In [7]:
df.columns

Index(['ticket_id', 'customer_name', 'customer_email', 'product', 'category',
       'issue_description', 'resolution_notes', 'priority', 'status',
       'channel', 'region', 'customer_age', 'customer_gender',
       'subscription_type', 'customer_tenure_months', 'previous_tickets',
       'customer_satisfaction_score', 'first_response_time_hours',
       'resolution_time_hours', 'ticket_created_date', 'ticket_resolved_date',
       'escalated', 'sla_breached', 'operating_system', 'browser',
       'payment_method', 'language', 'preferred_contact_time',
       'issue_complexity_score', 'customer_segment'],
      dtype='str')

In [8]:
clean_df = df[['product','category','issue_description', 'resolution_notes','operating_system','browser','payment_method']]

In [9]:
clean_df.head()

,product,category,issue_description,resolution_notes,operating_system,browser,payment_method
0,Web Portal,Account Suspension,The payment was deducted from my bank account ...,Data synchronization restored after backend se...,MacOS,Edge,PayPal
1,Mobile App,Performance Issue,I found a bug in the latest update affecting r...,Provided step-by-step troubleshooting instruct...,Windows,Firefox,PayPal
2,Web Portal,Performance Issue,The application crashes whenever I try to uplo...,Provided step-by-step troubleshooting instruct...,Windows,Safari,Bank Transfer
3,Payment Gateway,Subscription Cancellation,My subscription was cancelled without my reque...,Provided step-by-step troubleshooting instruct...,Windows,Chrome,Credit Card
4,Web Portal,Feature Request,The system is not syncing data across devices ...,We have reset the account credentials and advi...,Linux,NaN,Debit Card


In [10]:
clean_df[['product','category','issue_description', 'resolution_notes','operating_system','browser','payment_method']].isnull().any()

product              False
category             False
issue_description    False
resolution_notes     False
operating_system     False
browser               True
payment_method       False
dtype: bool

In [11]:
clean_df['browser'].isnull().sum()

np.int64(40023)

In [12]:
## As browser feature has more null droppgint that collumn also it's not a most dependent feature

clean_df = clean_df.drop( columns=['browser'])

In [13]:
clean_df.head()

,product,category,issue_description,resolution_notes,operating_system,payment_method
0,Web Portal,Account Suspension,The payment was deducted from my bank account ...,Data synchronization restored after backend se...,MacOS,PayPal
1,Mobile App,Performance Issue,I found a bug in the latest update affecting r...,Provided step-by-step troubleshooting instruct...,Windows,PayPal
2,Web Portal,Performance Issue,The application crashes whenever I try to uplo...,Provided step-by-step troubleshooting instruct...,Windows,Bank Transfer
3,Payment Gateway,Subscription Cancellation,My subscription was cancelled without my reque...,Provided step-by-step troubleshooting instruct...,Windows,Credit Card
4,Web Portal,Feature Request,The system is not syncing data across devices ...,We have reset the account credentials and advi...,Linux,Debit Card


In [14]:
clean_df.isnull().any()

product              False
category             False
issue_description    False
resolution_notes     False
operating_system     False
payment_method       False
dtype: bool

### Combain all columns to a single document to store in vector DB

In [15]:

clean_df["document_text"] = (
    "Product: " + clean_df["product"] + "\n\n" +
    "Category: " + clean_df["category"] + "\n\n" +
    "Issue Description:\n" + clean_df["issue_description"] + "\n\n" +
    "Resolution Notes:\n" + clean_df["resolution_notes"] 
)

In [16]:
clean_df.head()

,product,category,issue_description,resolution_notes,operating_system,payment_method,document_text
0,Web Portal,Account Suspension,The payment was deducted from my bank account ...,Data synchronization restored after backend se...,MacOS,PayPal,Product: Web Portal\n\nCategory: Account Suspe...
1,Mobile App,Performance Issue,I found a bug in the latest update affecting r...,Provided step-by-step troubleshooting instruct...,Windows,PayPal,Product: Mobile App\n\nCategory: Performance I...
2,Web Portal,Performance Issue,The application crashes whenever I try to uplo...,Provided step-by-step troubleshooting instruct...,Windows,Bank Transfer,Product: Web Portal\n\nCategory: Performance I...
3,Payment Gateway,Subscription Cancellation,My subscription was cancelled without my reque...,Provided step-by-step troubleshooting instruct...,Windows,Credit Card,Product: Payment Gateway\n\nCategory: Subscrip...
4,Web Portal,Feature Request,The system is not syncing data across devices ...,We have reset the account credentials and advi...,Linux,Debit Card,Product: Web Portal\n\nCategory: Feature Reque...


In [17]:
clean_df['document_text'][0]

'Product: Web Portal\n\nCategory: Account Suspension\n\nIssue Description:\nThe payment was deducted from my bank account but the transaction shows failed.\n\nResolution Notes:\nData synchronization restored after backend service restart.'

### Embedding 

In [24]:
import os
from dotenv import load_dotenv
load_dotenv()

True

###  As out document text is simple english and smaller sentences in size so we can use Free Embedding models
### I am chossing BAAI/bge-small-en-v1.5 which is hosted in hugging face

In [25]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7283.99it/s]


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [26]:
vector = embeddings.embed_query(
    "Payment failed after upgrade"
)

In [27]:
print(type(vector))
print(len(vector))

<class 'list'>
384


In [28]:
documents = clean_df["document_text"].tolist()
len(documents)

200000

In [29]:
documents_cleared = documents[:10000]
len(documents_cleared)

10000

### splitting in small bath and create embeddings as document size is too high

In [30]:
batch_size = 1000
vectors = []

for i in range(0, len(documents_cleared), batch_size):
    batch = documents_cleared[i : i + batch_size]
    print(f"Processing batch {i} ")
    batch_vectors = embeddings.embed_documents(batch)
    vectors.extend(batch_vectors)
    
    
print(f"Successfully generated {len(vectors)} embeddings.")

Processing batch 0 
Processing batch 1000 
Processing batch 2000 
Processing batch 3000 
Processing batch 4000 
Processing batch 5000 
Processing batch 6000 
Processing batch 7000 
Processing batch 8000 
Processing batch 9000 
Successfully generated 10000 embeddings.


In [31]:
len(vectors)

10000

### creating to pinecone index

In [32]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
pc


Pinecone(api_key='...CF4g', host='https://api.pinecone.io')

In [33]:
from pinecone import ServerlessSpec
index_name = "support-rag-openai"
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws",region="us-east-1")
    )

In [34]:
index= pc.Index(index_name)
index

Index(host='https://support-rag-openai-y3apscs.svc.aped-4627-b74a.pinecone.io')

In [35]:
subset_df = clean_df.head(10000).copy()

pinecone_vectors = []

for i, (idx, row) in enumerate(subset_df.iterrows()):
    pinecone_vectors.append({
        "id": str(idx),
        "values": vectors[i], 
        "metadata": {
            "product": str(row["product"]),
            "category": str(row["category"]),
            "operating_system": str(row["operating_system"]),
            "payment_method": str(row["payment_method"]),
            "text": str(row["document_text"])
        }
    })

print(len(pinecone_vectors))


10000


In [36]:
len(pinecone_vectors[0]["values"])

384

In [37]:
pinecone_vectors[0]["metadata"]

{'product': 'Web Portal',
 'category': 'Account Suspension',
 'operating_system': 'MacOS',
 'payment_method': 'PayPal',
 'text': 'Product: Web Portal\n\nCategory: Account Suspension\n\nIssue Description:\nThe payment was deducted from my bank account but the transaction shows failed.\n\nResolution Notes:\nData synchronization restored after backend service restart.'}

In [38]:
from rank_bm25 import BM25Okapi

### pushing the data to pinecode

In [ ]:
batch_size = 500

for i in range(0, len(pinecone_vectors), batch_size):

    batch = pinecone_vectors[i:i + batch_size]

    index.upsert(vectors=batch)

    print(
        f"Uploaded {i + len(batch)} of {len(pinecone_vectors)}"
    )

In [40]:
stats = index.describe_index_stats()
stats

DescribeIndexStatsResponse(dimension=384, total_vector_count=10000, metric='cosine', namespaces=1)

In [41]:
embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
import pickle

